# L08 · 演示数据获取与脚本化专家

本实验把讲义中选定的脚本化专家路线应用到一次完整的 banana-to-bowl 任务：

```text
演示来源选型 → 任务与抓取合同 → 七阶段专家
→ 内存中的 command/state trace → containment 证据
```

Notebook 会直接呈现任务合同、阶段逻辑、命令调度、实测轨迹和结果判定，同时复用共享控制器实现。

## 运行前准备

`ROBO_GENESIS_BACKEND=auto` 会在可用时选择已验证的 AMD backend，否则使用 CPU。设为 `cpu` 可以明确要求最低兼容 backend。`ROBO_GENESIS_RENDER=0` 会在不创建相机的情况下执行完整 rollout；在启动 kernel 前设为 `1`，则会要求八张 world-view 阶段图像。修改任一设置前都应重启 kernel。

CPU 路径是合法的 fallback，并不表示推荐优先使用 CPU；它的仿真吞吐通常低于已验证的 AMD 路径。

先预测：

1. 哪些信息属于 `TaskSpec`，哪些属于 `GraspProfile`？
2. 为什么场景静置发生在七个动作阶段之前？
3. 为什么 commanded action 与 measured state 可以都具有 `(T, 9)` shape，却不必相等？
4. 为什么 bowl containment 必须同时检查水平位置和 below-rim depth？

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np

from robo_genesis.build_scene import build_scene
from robo_genesis.course_manifest import load_course_manifest
from robo_genesis.course_utils import environment_report, notebook_mode, select_backend, to_numpy
from robo_genesis.grasp_demo import (
    GraspProfile,
    MOVE_MAX_DQ,
    MOVE_MIN_STEPS,
    TaskSpec,
    check_success,
    run_pick_place,
)
from robo_genesis.scene_config import WORLD_CAM_RES

lesson = load_course_manifest().lesson('L08')
assert lesson.slug == 'demonstration-acquisition-and-scripted-experts'
assert lesson.duration_minutes == 120
assert lesson.hardware.value == 'gpu-recommended'
assert lesson.status.value == 'planned'

backend_mode = os.environ.get('ROBO_GENESIS_BACKEND', 'auto').strip().lower()
if backend_mode not in {'auto', 'cpu'}:
    raise ValueError("ROBO_GENESIS_BACKEND must be 'auto' or 'cpu'")
render_value = os.environ.get('ROBO_GENESIS_RENDER', '0').strip()
if render_value not in {'0', '1'}:
    raise ValueError("ROBO_GENESIS_RENDER must be '0' or '1'")
render_enabled = render_value == '1'

runtime = notebook_mode('l08-scripted-expert', show_viewer=False)
environment = environment_report()

import genesis as gs

backend = gs.cpu if backend_mode == 'cpu' else select_backend(prefer_rocm=True)
gs.init(backend=backend, seed=0, precision='32', logging_level='warning')

if getattr(gs, 'amdgpu', None) is not None and gs.backend == gs.amdgpu:
    actual_backend = 'amdgpu'
elif gs.backend == gs.cpu:
    actual_backend = 'cpu'
else:
    actual_backend = str(gs.backend)

print('Genesis:', environment['genesis_world'])
print('requested backend:', backend_mode)
print('actual backend:', actual_backend)
print('render enabled:', render_enabled)
print('output directory:', runtime['output_dir'].resolve())


## 明确专家合同

`TaskSpec` 说明要完成什么：抓取 banana、放入 bowl，并使用给定的水平容差。选中的 `GraspProfile` 说明怎样抓取这个物体：夹爪 yaw、hand height 策略和闭合力。

下面的紧凑阶段表是可直接阅读的状态机骨架；共享 `run_pick_place()` 实现仍是行为真相源。

In [ ]:
PHASES = (
    ('pregrasp', 'pose above object', 'IK + collision-checked plan'),
    ('descend', 'fixed-xy vertical approach', 'IK at descending z waypoints'),
    ('grasp', 'establish and hold contact', 'arm position + finger force'),
    ('lift', 'clear the tabletop', 'bounded-increment arm targets'),
    ('transport', 'move above the bowl', 'bounded-increment arm targets'),
    ('release', 'let the object enter the bowl', 'open finger position target'),
    ('retreat', 'move hand away and settle', 'direct retreat + settle'),
)
EXPECTED_FRAME_TAGS = (
    '00_start',
    '01_pregrasp',
    '02_reach',
    '03_grasp',
    '04_lift',
    '05_above_target',
    '06_release',
    '07_done',
)

task = TaskSpec(
    pick_object='011_banana',
    place_target='024_bowl',
    success_tol=0.06,
)
profile = task.grasp_profile()
contract_checks = {
    'seven_ordered_phases': tuple(row[0] for row in PHASES)
    == ('pregrasp', 'descend', 'grasp', 'lift', 'transport', 'release', 'retreat'),
    'banana_to_bowl_task': task.pick_object == '011_banana'
    and task.place_target == '024_bowl',
    'grasp_profile_type': isinstance(profile, GraspProfile),
    'finite_profile': np.isfinite(
        [profile.yaw_offset, profile.grasp_hand_z, profile.close_force]
    ).all(),
}

for index, (phase, target, control) in enumerate(PHASES, start=1):
    print(f'{index}. {phase:<10} target={target:<31} control={control}')
print('task:', task)
print('profile:', profile)
for name, passed in contract_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")
if not all(contract_checks.values()):
    raise AssertionError(contract_checks)


## 验证 commanded-target schedule

在 lift 和 transport 阶段，专家从实测 `q_start` 线性插值到 IK goal。Waypoint 数量为 `max(MOVE_MIN_STEPS, ceil(Δq∞ / MOVE_MAX_DQ))`，因此相邻 commanded arm target 的 infinity-norm 变化存在上界。

第二组计算是单变量练习：先预测更小的 `CANDIDATE_MAX_DQ` 会带来什么变化，再比较 waypoint 数量和最大 command step。这里验证的只是 command schedule，不能据此宣称某个 measured acceleration 或更高的抓取成功率。

In [ ]:
q_start = np.array([0.00, -0.30, 0.10, -1.80, 0.05, 1.55, 0.70], dtype=float)
q_goal = np.array([0.18, -0.08, 0.32, -1.42, -0.11, 1.82, 0.54], dtype=float)

delta_q_inf = float(np.max(np.abs(q_goal - q_start)))
waypoint_count = max(MOVE_MIN_STEPS, int(np.ceil(delta_q_inf / MOVE_MAX_DQ)))
fractions = np.arange(1, waypoint_count + 1, dtype=float)[:, None] / waypoint_count
command_schedule = q_start + (q_goal - q_start) * fractions
schedule_with_start = np.vstack([q_start, command_schedule])
max_command_step = float(np.max(np.abs(np.diff(schedule_with_start, axis=0))))

CANDIDATE_MAX_DQ = 0.003
candidate_waypoint_count = max(
    MOVE_MIN_STEPS, int(np.ceil(delta_q_inf / CANDIDATE_MAX_DQ))
)
candidate_fractions = (
    np.arange(1, candidate_waypoint_count + 1, dtype=float)[:, None]
    / candidate_waypoint_count
)
candidate_schedule = q_start + (q_goal - q_start) * candidate_fractions
candidate_with_start = np.vstack([q_start, candidate_schedule])
candidate_max_step = float(np.max(np.abs(np.diff(candidate_with_start, axis=0))))

schedule_checks = {
    'waypoint_formula': waypoint_count
    == max(MOVE_MIN_STEPS, int(np.ceil(delta_q_inf / MOVE_MAX_DQ))),
    'baseline_endpoint': np.allclose(command_schedule[-1], q_goal),
    'baseline_step_bound': max_command_step <= MOVE_MAX_DQ + 1e-12,
    'candidate_endpoint': np.allclose(candidate_schedule[-1], q_goal),
    'candidate_step_bound': candidate_max_step <= CANDIDATE_MAX_DQ + 1e-12,
    'smaller_bound_uses_no_fewer_waypoints': candidate_waypoint_count >= waypoint_count,
}

print(f'Δq∞: {delta_q_inf:.6f} rad')
print(
    f'baseline max_dq={MOVE_MAX_DQ:.6f}: '
    f'{waypoint_count} waypoints, max step={max_command_step:.6f} rad'
)
print(
    f'candidate max_dq={CANDIDATE_MAX_DQ:.6f}: '
    f'{candidate_waypoint_count} waypoints, max step={candidate_max_step:.6f} rad'
)
for name, passed in schedule_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")
if not all(schedule_checks.values()):
    raise AssertionError(schedule_checks)


## 构建固定任务场景

共享 builder 会复用 L07 的桌面、Franka、YCB 物体、控制器配置和稳定资产路径。本讲保持任务固定，并传入 `scene_dr=None`；domain randomization 留到 L11。

禁用渲染时不创建相机。启用渲染时只请求固定 world camera，因为这里要观察阶段 montage，而不是再次讲解相机。

In [ ]:
bundle = build_scene(
    show_viewer=False,
    n_envs=1,
    add_world_cam=render_enabled,
    add_wrist_cam=False,
    add_video_cam=False,
    draw_world_frame=False,
    scene_dr=None,
)
initial_qpos = to_numpy(bundle.franka.get_qpos()).astype(float).reshape(-1)

build_checks = {
    'unbatched_franka_qpos': initial_qpos.shape == (9,),
    'franka_qpos_finite': np.isfinite(initial_qpos).all(),
    'task_entities_present': task.pick_object in bundle.ycb
    and task.place_target in bundle.ycb,
    'requested_world_camera': (bundle.world_cam is not None) == render_enabled,
    'no_wrist_camera': bundle.wrist_cam is None,
    'no_video_camera': bundle.video_cam is None,
}
for name, passed in build_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")
if not all(build_checks.values()):
    raise AssertionError(build_checks)

print('Franka qpos shape:', initial_qpos.shape)
print('task entities:', task.pick_object, '→', task.place_target)
print('world camera:', 'created' if render_enabled else 'SKIP — not created')


## 执行一次脚本化 rollout，并把 trace 保留在内存中

轻量 recorder 会在发送对应命令并推进仿真之前，读取 Franka 的实测 qpos。它不写文件，也不定义 dataset schema：

```text
state_t  = measured [arm_q(7), finger_q(2)]
action_t = commanded [arm_target(7), finger_target(2)]
```

L09 会继续增加 timestamp、image sampling、episode boundary 和持久化存储。本单元只执行一次固定 rollout。

In [ ]:
class TraceRecorder:
    def __init__(self, scene_bundle):
        self.bundle = scene_bundle
        self.states = []
        self.actions = []

    def on_step(self, action):
        measured_qpos = to_numpy(self.bundle.franka.get_qpos()).astype(float).reshape(-1)
        commanded_action = to_numpy(action).astype(float).reshape(-1)
        self.states.append(measured_qpos.copy())
        self.actions.append(commanded_action.copy())


trace = TraceRecorder(bundle)
rollout_success, stage_frames = run_pick_place(
    bundle,
    task,
    save_frames=render_enabled,
    recorder=trace,
)
state_trace = np.stack(trace.states)
action_trace = np.stack(trace.actions)

print('rollout success:', rollout_success)
print('state trace:', state_trace.shape, state_trace.dtype)
print('action trace:', action_trace.shape, action_trace.dtype)
print('captured stage frames:', len(stage_frames))


## 区分 trace 证据与任务证据

相同的 `(T, 9)` shape 只能证明两组序列在长度和维度上对齐，并不表示数值相等；有限的控制器响应和接触都可能使实测 qpos 偏离请求的 target。

对于 bowl 结果，水平分量检查 banana 是否位于允许的开口半径内，垂直分量检查 banana AABB bottom 是否至少低于 bowl rim 当前使用的 1 cm margin。两项都必须通过，而且它们的合取结果必须与共享 `check_success()` predicate 一致。

In [ ]:
trace_delta = np.abs(action_trace - state_trace)
trace_checks = {
    'same_nonzero_length': len(state_trace) == len(action_trace) > 0,
    'state_shape': state_trace.ndim == 2 and state_trace.shape[1] == 9,
    'action_shape': action_trace.ndim == 2 and action_trace.shape[1] == 9,
    'floating_arrays': np.issubdtype(state_trace.dtype, np.floating)
    and np.issubdtype(action_trace.dtype, np.floating),
    'finite_arrays': np.isfinite(state_trace).all() and np.isfinite(action_trace).all(),
    'command_state_difference_observed': bool(np.max(trace_delta) > 0.0),
}

banana = bundle.ycb[task.pick_object]
bowl = bundle.ycb[task.place_target]
banana_pos = to_numpy(banana.get_pos()).astype(float).reshape(-1)
bowl_pos = to_numpy(bowl.get_pos()).astype(float).reshape(-1)
banana_aabb = to_numpy(banana.get_AABB()).astype(float).reshape(2, 3)
bowl_aabb = to_numpy(bowl.get_AABB()).astype(float).reshape(2, 3)

horizontal_distance = float(np.linalg.norm(banana_pos[:2] - bowl_pos[:2]))
bowl_rim_radius = 0.5 * float(
    min(bowl_aabb[1, 0] - bowl_aabb[0, 0], bowl_aabb[1, 1] - bowl_aabb[0, 1])
)
allowed_radius = min(task.success_tol, bowl_rim_radius)
within_footprint = horizontal_distance < allowed_radius
BOWL_RIM_MARGIN = 0.01
object_aabb_bottom_z = float(banana_aabb[0, 2])
bowl_rim_z = float(bowl_aabb[1, 2])
inside_bowl = object_aabb_bottom_z < bowl_rim_z - BOWL_RIM_MARGIN
shared_success = check_success(bundle, task)

outcome_checks = {
    'within_footprint': within_footprint,
    'inside_bowl': inside_bowl,
    'expanded_predicate_matches_shared_check':
    bool(within_footprint and inside_bowl) == shared_success,
    'rollout_result_matches_shared_check': bool(rollout_success) == shared_success,
    'task_completed': bool(rollout_success),
}

print(f'horizontal distance: {horizontal_distance:.6f} m')
print(f'allowed radius:      {allowed_radius:.6f} m')
print(f'object AABB bottom:  {object_aabb_bottom_z:.6f} m')
print(f'bowl rim:            {bowl_rim_z:.6f} m')
for name, passed in {**trace_checks, **outcome_checks}.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")


## 检查可选的阶段 montage

启用渲染时，下一个单元会验证并显示静置后的起点，以及每个阶段结束后的一张 world-view 图像。Tag 对应共享实现中的观察时刻：`02_reach` 位于 descend 之后，`05_above_target` 位于 transport 之后，`07_done` 位于 retreat 和最终静置之后。

禁用渲染时，该单元会确认没有创建相机或 frame，并明确报告 `SKIP`。图像用于帮助解释阶段顺序，最终是否通过仍由 trace 与 containment 检查决定。

## 检查点与扩展

运行最终单元前，请用自己的话解释完整证据链：

1. `TaskSpec` 与 `GraspProfile` 怎样区分任务意图和抓取假设？
2. 每个阶段分别使用了什么 primitive 和控制模式？
3. Rate-schedule 计算证明了什么，又有哪些物理行为没有被它证明？
4. 如果只有 `within_footprint` 未通过，应该优先诊断结果的哪一部分？

课后可以选择 lemon 或 plum、多个独立 seed，或者单独运行 `motion_probe.py --compare`。不要把这一次成功 rollout 解释成专家成功率。

In [ ]:
visual_status = 'SKIP — ROBO_GENESIS_RENDER=0; no camera or stage frames were created'
visual_checks = {
    'camera_absent_when_disabled': not render_enabled and bundle.world_cam is None,
    'frames_absent_when_disabled': not render_enabled and len(stage_frames) == 0,
}

if render_enabled:
    frame_tags = tuple(tag for tag, _ in stage_frames)
    frame_images = [to_numpy(image) for _, image in stage_frames]
    width, height = WORLD_CAM_RES
    visual_checks = {
        'world_camera_present': bundle.world_cam is not None,
        'eight_stage_frames': len(frame_images) == 8,
        'ordered_stage_tags': frame_tags == EXPECTED_FRAME_TAGS,
        'rgb_shapes': all(image.shape == (height, width, 3) for image in frame_images),
        'rgb_dtype': all(image.dtype == np.uint8 for image in frame_images),
        'finite_pixels': all(np.isfinite(image).all() for image in frame_images),
        'within_frame_variation': all(bool(np.std(image) > 0.0) for image in frame_images),
        'sequence_pixel_change': any(
            not np.array_equal(left, right)
            for left, right in zip(frame_images, frame_images[1:])
        ),
    }
    if all(visual_checks.values()):
        figure, axes = plt.subplots(2, 4, figsize=(16, 8))
        for axis, tag, image in zip(axes.ravel(), frame_tags, frame_images):
            axis.imshow(image)
            axis.set_title(tag)
            axis.axis('off')
        figure.tight_layout()
        plt.show()
        visual_status = 'PASSED — start + seven phase images validated'

final_checks = {
    'runtime_contract': environment['genesis_world'] == '1.3.3'
    and actual_backend in {'cpu', 'amdgpu'}
    and (backend_mode != 'cpu' or actual_backend == 'cpu'),
    'manifest_contract': lesson.status.value == 'planned'
    and lesson.duration_minutes == 120,
    'expert_contract': all(contract_checks.values()),
    'rate_schedule': all(schedule_checks.values()),
    'scene_build': all(build_checks.values()),
    'trace_evidence': all(trace_checks.values()),
    'outcome_evidence': all(outcome_checks.values()),
    'visual_branch': all(visual_checks.values()),
}
for name, passed in final_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")
failed = [name for name, passed in final_checks.items() if not passed]
if failed:
    raise AssertionError('L08 checks failed: ' + ', '.join(failed))

print('visual evidence:', visual_status)
print('L08 CHECK: PASSED')
